In [ ]:
import kagglehub

# Download latest version of the dataset
path = kagglehub.dataset_download("renukasiriwardhana/fabric-classification-dataset")
print("Path to dataset files:", path)

In [ ]:
import os

print("=== Folder structure ===")
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root) if os.path.basename(root) else root}/")
    if files and level <= 3:
        print(f"{' ' * 2 * (level+1)}({len(files)} files)")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
import shutil
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
import os

# Clean up any partial split from the failed attempt
import shutil as sh
split_base = '/kaggle/working/split_dataset'
if os.path.exists(split_base):
    sh.rmtree(split_base)
    print("Cleaned up partial split")

classes = sorted([d for d in os.listdir(dataset_root)
                   if os.path.isdir(os.path.join(dataset_root, d))])
print("Classes found:", classes)

for split in ['train', 'val', 'test']:
    for c in classes:
        os.makedirs(f'{split_base}/{split}/{c}', exist_ok=True)

for c in classes:
    src_folder = os.path.join(dataset_root, c)
    imgs = [f for f in os.listdir(src_folder) if f.lower().endswith(('.png','.jpg','.jpeg'))]
    random.shuffle(imgs)

    n = len(imgs)
    train_end = int(0.7 * n)
    val_end = int(0.85 * n)

    for i, img in enumerate(imgs):
        src = os.path.join(src_folder, img)
        if i < train_end:
            dst = f'{split_base}/train/{c}/{img}'
        elif i < val_end:
            dst = f'{split_base}/val/{c}/{img}'
        else:
            dst = f'{split_base}/test/{c}/{img}'

        # Use symlink instead of copy - uses almost zero disk space
        os.symlink(src, dst)

train_dir = f'{split_base}/train'
val_dir = f'{split_base}/val'
test_dir = f'{split_base}/test'
print("✅ Split created successfully using symlinks (no extra disk space used)!")
print(f"Train: {train_dir}\nVal: {val_dir}\nTest: {test_dir}")

# Verify counts
for split in ['train', 'val', 'test']:
    total = sum(len(os.listdir(f'{split_base}/{split}/{c}')) for c in classes)
    print(f"{split}: {total} images")

In [ ]:
print("Loading datasets...")
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, image_size=(224, 224), batch_size=32
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir, image_size=(224, 224), batch_size=32
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir, image_size=(224, 224), batch_size=32, shuffle=False
)

class_names = train_ds.class_names
print("Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

In [ ]:
# Extract labels before prefetching train_ds
train_labels = np.concatenate([y for x, y in train_ds], axis=0)

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = dict(enumerate(class_weights_array))
print("Class weights:", class_weights)

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, image_size=(224, 224), batch_size=32
).prefetch(buffer_size=AUTOTUNE)

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.15),
], name="data_augmentation")

In [ ]:
preprocess = tf.keras.applications.densenet.preprocess_input

base_model = tf.keras.applications.DenseNet121(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)
base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = layers.Lambda(preprocess, name="preprocess_input")(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)

x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)

outputs = layers.Dense(len(class_names), activation="softmax")(x)
model = models.Model(inputs, outputs)

model.summary()

In [ ]:
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1)
early_stopping = EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True, verbose=1)
checkpoint = ModelCheckpoint("/kaggle/working/best_densenet_fabric.keras",
                              monitor="val_loss", save_best_only=True, verbose=1)

In [ ]:
print("\n--- Stage 1: Training the new layers only ---")
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=[reduce_lr, early_stopping, checkpoint],
    class_weight=class_weights
)

In [ ]:
print("\n--- Stage 2: Fine-tuning top layers of DenseNet ---")
base_model.trainable = True

for layer in base_model.layers[:-60]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=[reduce_lr, early_stopping, checkpoint],
    class_weight=class_weights
)

print("Training Completed Successfully!")

In [ ]:
print("Loading the best saved model...")
custom_objects = {'preprocess_input': tf.keras.applications.densenet.preprocess_input}
best_model = tf.keras.models.load_model("/kaggle/working/best_densenet_fabric.keras", custom_objects=custom_objects)

test_loss, test_acc = best_model.evaluate(test_ds)
print(f"\nFinal Test Accuracy: {test_acc*100:.2f}%")

y_pred_probs = best_model.predict(test_ds)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.concatenate([y for x, y in test_ds], axis=0)

print("\n=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Fabric Classification')
plt.ylabel('True Fabric Class')
plt.xlabel('Predicted Fabric Class')
plt.show()

class_accuracies = cm.diagonal() / cm.sum(axis=1)
accuracy_dict = {class_names[i]: acc * 100 for i, acc in enumerate(class_accuracies)}
sorted_classes = sorted(accuracy_dict.items(), key=lambda x: x[1], reverse=True)

print("\n--- Model Performance by Fabric Class ---")
print("\nTop Performing Classes (>=80%):")
for fabric, acc in sorted_classes:
    if acc >= 80:
        print(f"  - {fabric}: {acc:.2f}%")

print("\nWeaker Performing Classes (<80%):")
for fabric, acc in reversed(sorted_classes):
    if acc < 80:
        print(f"  - {fabric}: {acc:.2f}%")

In [ ]:
acc = history1.history['accuracy'] + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss = history1.history['loss'] + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']

epochs_range = range(len(acc))

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.axvline(x=len(history1.history['accuracy'])-1, color='r', linestyle='--', label='Start of Phase 2')
plt.title('Phase 1 vs Phase 2: Accuracy')
plt.legend(loc='lower right')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.axvline(x=len(history1.history['loss'])-1, color='r', linestyle='--', label='Start of Phase 2')
plt.title('Phase 1 vs Phase 2: Loss')
plt.legend(loc='upper right')
plt.show()

print(f"\nPhase 1 Epochs: {len(history1.history['accuracy'])}")
print(f"Phase 2 Epochs: {len(history2.history['accuracy'])}")

In [ ]:
from tensorflow.keras.preprocessing import image
from IPython.display import display
import ipywidgets as widgets

# Kaggle notebooks don't have Colab's files.upload() - use file path instead
# Upload an image via Kaggle's "Add Data" or use a test image from your test set

def predict_fabric(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    plt.figure(figsize=(4,4))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    predictions = best_model.predict(img_array)
    predicted_class = class_names[np.argmax(predictions[0])]
    confidence = np.max(predictions[0]) * 100

    print(f"Predicted: {predicted_class} ({confidence:.2f}%)")

# Example usage - replace with an actual test image path:
# predict_fabric(test_dir + '/Silk/some_image.jpg')

In [ ]:
import os

# Create output directory
save_dir = '/kaggle/working/final_model'
os.makedirs(save_dir, exist_ok=True)

# 1. Save as native Keras format (best for Python/TensorFlow apps, backend servers)
best_model.save(f'{save_dir}/fabric_classifier.keras')
print("✅ Saved .keras format")

# 2. Save as SavedModel format (best for TensorFlow Serving, production APIs)
best_model.export(f'{save_dir}/saved_model')
print("✅ Saved SavedModel format")

# 3. Save class names too - app eken use karanna ona
import json
class_info = {
    "classes": class_names,
    "input_size": [224, 224],
    "preprocessing": "densenet"
}
with open(f'{save_dir}/class_info.json', 'w') as f:
    json.dump(class_info, f, indent=2)
print("✅ Saved class_info.json")

print("\nFiles saved:")
for root, dirs, files in os.walk(save_dir):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# Install tensorflowjs converter
!pip install tensorflowjs -q

!tensorflowjs_converter --input_format=tf_saved_model \
    {save_dir}/saved_model \
    {save_dir}/tfjs_model

print("✅ TensorFlow.js model saved")

In [ ]:
import shutil

# Zip the entire final_model folder for easy download
shutil.make_archive('/kaggle/working/fabric_model_package', 'zip', save_dir)
print("✅ Zipped: /kaggle/working/fabric_model_package.zip")
print(f"Size: {os.path.getsize('/kaggle/working/fabric_model_package.zip') / (1024*1024):.2f} MB")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np
import io
from PIL import Image

# Create upload button
upload_widget = widgets.FileUpload(
    accept='.jpg,.jpeg,.png',
    multiple=False,
    description='Upload Image'
)

output = widgets.Output()

def on_upload_change(change):
    with output:
        clear_output()

        if len(upload_widget.value) == 0:
            return

        uploaded_file = list(upload_widget.value.values())[0] if isinstance(upload_widget.value, dict) else upload_widget.value[0]
        content = uploaded_file['content'] if isinstance(uploaded_file, dict) else uploaded_file.content

        img = Image.open(io.BytesIO(content)).convert('RGB')
        img_resized = img.resize((224, 224))

        plt.figure(figsize=(5, 5))
        plt.imshow(img_resized)
        plt.axis('off')
        plt.title("Uploaded Image")
        plt.show()

        img_array = np.array(img_resized)
        img_array = np.expand_dims(img_array, axis=0)

        predictions = best_model.predict(img_array)
        predicted_index = np.argmax(predictions[0])
        predicted_class = class_names[predicted_index]
        confidence = np.max(predictions[0]) * 100

        print(f"\nPredicted Fabric: {predicted_class}")
        print(f"Confidence: {confidence:.2f}%")

        top3_idx = np.argsort(predictions[0])[-3:][::-1]
        print("\nTop-3 Predictions:")
        for idx in top3_idx:
            pct = predictions[0][idx] * 100
            print(f"  {class_names[idx]}: {pct:.2f}%")

upload_widget.observe(on_upload_change, names='value')

print("Click 'Upload Image' below and select a fabric photo from your computer:")
display(upload_widget, output)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np
import io
from PIL import Image

# Create upload button
upload_widget = widgets.FileUpload(
    accept='.jpg,.jpeg,.png',
    multiple=False,
    description='Upload Image'
)

output = widgets.Output()

def on_upload_change(change):
    with output:
        clear_output()

        if len(upload_widget.value) == 0:
            return

        uploaded_file = list(upload_widget.value.values())[0] if isinstance(upload_widget.value, dict) else upload_widget.value[0]
        content = uploaded_file['content'] if isinstance(uploaded_file, dict) else uploaded_file.content

        img = Image.open(io.BytesIO(content)).convert('RGB')
        img_resized = img.resize((224, 224))

        plt.figure(figsize=(5, 5))
        plt.imshow(img_resized)
        plt.axis('off')
        plt.title("Uploaded Image")
        plt.show()

        img_array = np.array(img_resized)
        img_array = np.expand_dims(img_array, axis=0)

        predictions = best_model.predict(img_array)
        predicted_index = np.argmax(predictions[0])
        predicted_class = class_names[predicted_index]
        confidence = np.max(predictions[0]) * 100

        print(f"\nPredicted Fabric: {predicted_class}")
        print(f"Confidence: {confidence:.2f}%")

        top3_idx = np.argsort(predictions[0])[-3:][::-1]
        print("\nTop-3 Predictions:")
        for idx in top3_idx:
            pct = predictions[0][idx] * 100
            print(f"  {class_names[idx]}: {pct:.2f}%")

upload_widget.observe(on_upload_change, names='value')

print("Click 'Upload Image' below and select a fabric photo from your computer:")
display(upload_widget, output)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np
import io
from PIL import Image

# Create upload button
upload_widget = widgets.FileUpload(
    accept='.jpg,.jpeg,.png',
    multiple=False,
    description='Upload Image'
)

output = widgets.Output()

def on_upload_change(change):
    with output:
        clear_output()

        if len(upload_widget.value) == 0:
            return

        uploaded_file = list(upload_widget.value.values())[0] if isinstance(upload_widget.value, dict) else upload_widget.value[0]
        content = uploaded_file['content'] if isinstance(uploaded_file, dict) else uploaded_file.content

        img = Image.open(io.BytesIO(content)).convert('RGB')
        img_resized = img.resize((224, 224))

        plt.figure(figsize=(5, 5))
        plt.imshow(img_resized)
        plt.axis('off')
        plt.title("Uploaded Image")
        plt.show()

        img_array = np.array(img_resized)
        img_array = np.expand_dims(img_array, axis=0)

        predictions = best_model.predict(img_array)
        predicted_index = np.argmax(predictions[0])
        predicted_class = class_names[predicted_index]
        confidence = np.max(predictions[0]) * 100

        print(f"\nPredicted Fabric: {predicted_class}")
        print(f"Confidence: {confidence:.2f}%")

        top3_idx = np.argsort(predictions[0])[-3:][::-1]
        print("\nTop-3 Predictions:")
        for idx in top3_idx:
            pct = predictions[0][idx] * 100
            print(f"  {class_names[idx]}: {pct:.2f}%")

upload_widget.observe(on_upload_change, names='value')

print("Click 'Upload Image' below and select a fabric photo from your computer:")
display(upload_widget, output)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np
import io
from PIL import Image

# Create upload button
upload_widget = widgets.FileUpload(
    accept='.jpg,.jpeg,.png',
    multiple=False,
    description='Upload Image'
)

output = widgets.Output()

def on_upload_change(change):
    with output:
        clear_output()

        if len(upload_widget.value) == 0:
            return

        uploaded_file = list(upload_widget.value.values())[0] if isinstance(upload_widget.value, dict) else upload_widget.value[0]
        content = uploaded_file['content'] if isinstance(uploaded_file, dict) else uploaded_file.content

        img = Image.open(io.BytesIO(content)).convert('RGB')
        img_resized = img.resize((224, 224))

        plt.figure(figsize=(5, 5))
        plt.imshow(img_resized)
        plt.axis('off')
        plt.title("Uploaded Image")
        plt.show()

        img_array = np.array(img_resized)
        img_array = np.expand_dims(img_array, axis=0)

        predictions = best_model.predict(img_array)
        predicted_index = np.argmax(predictions[0])
        predicted_class = class_names[predicted_index]
        confidence = np.max(predictions[0]) * 100

        print(f"\nPredicted Fabric: {predicted_class}")
        print(f"Confidence: {confidence:.2f}%")

        top3_idx = np.argsort(predictions[0])[-3:][::-1]
        print("\nTop-3 Predictions:")
        for idx in top3_idx:
            pct = predictions[0][idx] * 100
            print(f"  {class_names[idx]}: {pct:.2f}%")

upload_widget.observe(on_upload_change, names='value')

print("Click 'Upload Image' below and select a fabric photo from your computer:")
display(upload_widget, output)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np
import io
from PIL import Image

# Create upload button
upload_widget = widgets.FileUpload(
    accept='.jpg,.jpeg,.png',
    multiple=False,
    description='Upload Image'
)

output = widgets.Output()

def on_upload_change(change):
    with output:
        clear_output()

        if len(upload_widget.value) == 0:
            return

        uploaded_file = list(upload_widget.value.values())[0] if isinstance(upload_widget.value, dict) else upload_widget.value[0]
        content = uploaded_file['content'] if isinstance(uploaded_file, dict) else uploaded_file.content

        img = Image.open(io.BytesIO(content)).convert('RGB')
        img_resized = img.resize((224, 224))

        plt.figure(figsize=(5, 5))
        plt.imshow(img_resized)
        plt.axis('off')
        plt.title("Uploaded Image")
        plt.show()

        img_array = np.array(img_resized)
        img_array = np.expand_dims(img_array, axis=0)

        predictions = best_model.predict(img_array)
        predicted_index = np.argmax(predictions[0])
        predicted_class = class_names[predicted_index]
        confidence = np.max(predictions[0]) * 100

        print(f"\nPredicted Fabric: {predicted_class}")
        print(f"Confidence: {confidence:.2f}%")

        top3_idx = np.argsort(predictions[0])[-3:][::-1]
        print("\nTop-3 Predictions:")
        for idx in top3_idx:
            pct = predictions[0][idx] * 100
            print(f"  {class_names[idx]}: {pct:.2f}%")

upload_widget.observe(on_upload_change, names='value')

print("Click 'Upload Image' below and select a fabric photo from your computer:")
display(upload_widget, output)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np
import io
from PIL import Image

# Create upload button
upload_widget = widgets.FileUpload(
    accept='.jpg,.jpeg,.png',
    multiple=False,
    description='Upload Image'
)

output = widgets.Output()

def on_upload_change(change):
    with output:
        clear_output()

        if len(upload_widget.value) == 0:
            return

        uploaded_file = list(upload_widget.value.values())[0] if isinstance(upload_widget.value, dict) else upload_widget.value[0]
        content = uploaded_file['content'] if isinstance(uploaded_file, dict) else uploaded_file.content

        img = Image.open(io.BytesIO(content)).convert('RGB')
        img_resized = img.resize((224, 224))

        plt.figure(figsize=(5, 5))
        plt.imshow(img_resized)
        plt.axis('off')
        plt.title("Uploaded Image")
        plt.show()

        img_array = np.array(img_resized)
        img_array = np.expand_dims(img_array, axis=0)

        predictions = best_model.predict(img_array)
        predicted_index = np.argmax(predictions[0])
        predicted_class = class_names[predicted_index]
        confidence = np.max(predictions[0]) * 100

        print(f"\nPredicted Fabric: {predicted_class}")
        print(f"Confidence: {confidence:.2f}%")

        top3_idx = np.argsort(predictions[0])[-3:][::-1]
        print("\nTop-3 Predictions:")
        for idx in top3_idx:
            pct = predictions[0][idx] * 100
            print(f"  {class_names[idx]}: {pct:.2f}%")

upload_widget.observe(on_upload_change, names='value')

print("Click 'Upload Image' below and select a fabric photo from your computer:")
display(upload_widget, output)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np
import io
from PIL import Image

# Create upload button
upload_widget = widgets.FileUpload(
    accept='.jpg,.jpeg,.png',
    multiple=False,
    description='Upload Image'
)

output = widgets.Output()

def on_upload_change(change):
    with output:
        clear_output()

        if len(upload_widget.value) == 0:
            return

        uploaded_file = list(upload_widget.value.values())[0] if isinstance(upload_widget.value, dict) else upload_widget.value[0]
        content = uploaded_file['content'] if isinstance(uploaded_file, dict) else uploaded_file.content

        img = Image.open(io.BytesIO(content)).convert('RGB')
        img_resized = img.resize((224, 224))

        plt.figure(figsize=(5, 5))
        plt.imshow(img_resized)
        plt.axis('off')
        plt.title("Uploaded Image")
        plt.show()

        img_array = np.array(img_resized)
        img_array = np.expand_dims(img_array, axis=0)

        predictions = best_model.predict(img_array)
        predicted_index = np.argmax(predictions[0])
        predicted_class = class_names[predicted_index]
        confidence = np.max(predictions[0]) * 100

        print(f"\nPredicted Fabric: {predicted_class}")
        print(f"Confidence: {confidence:.2f}%")

        top3_idx = np.argsort(predictions[0])[-3:][::-1]
        print("\nTop-3 Predictions:")
        for idx in top3_idx:
            pct = predictions[0][idx] * 100
            print(f"  {class_names[idx]}: {pct:.2f}%")

upload_widget.observe(on_upload_change, names='value')

print("Click 'Upload Image' below and select a fabric photo from your computer:")
display(upload_widget, output)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

# Show a sample from training data
sample_class = 'Cotton'  # try any class
sample_folder = f'{train_dir}/{sample_class}'
sample_img = os.listdir(sample_folder)[0]
img = Image.open(f'{sample_folder}/{sample_img}')

plt.imshow(img)
plt.title(f"Training sample: {sample_class}")
plt.axis('off')
plt.show()

In [ ]:
test_loss, test_acc = best_model.evaluate(test_ds)
print(f"Test Accuracy: {test_acc*100:.2f}%")

In [ ]:
print("class_names variable:", class_names)

# Compare with actual folder order
import os
actual_classes = sorted(os.listdir(train_dir))
print("Actual folder order:", actual_classes)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

classes_to_check = ['Silk', 'Cotton', 'Polyester', 'Denim']

fig, axes = plt.subplots(1, len(classes_to_check), figsize=(16, 4))

for i, cls in enumerate(classes_to_check):
    folder = f'{train_dir}/{cls}'
    sample_img = os.listdir(folder)[0]
    img = Image.open(f'{folder}/{sample_img}')
    axes[i].imshow(img)
    axes[i].set_title(cls)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from PIL import Image

def prepare_image_for_prediction(pil_image, crop_ratio=0.5):
    """
    Simulates the 'close-up zoom' style of the training dataset
    by cropping the center portion of the image and resizing it.
    This bridges the domain gap between training data (zoomed) 
    and real-world test photos (full view).
    """
    w, h = pil_image.size
    # Crop center square region (removes background/edges, focuses on texture)
    crop_w, crop_h = int(w * crop_ratio), int(h * crop_ratio)
    left = (w - crop_w) // 2
    top = (h - crop_h) // 2
    cropped = pil_image.crop((left, top, left + crop_w, top + crop_h))

    # Resize to model input size
    resized = cropped.resize((224, 224))
    return resized

In [ ]:
def predict_with_tta(pil_image, model, class_names, num_crops=5):
    """
    Takes multiple zoomed crops from different regions of the image
    and averages predictions - more robust to domain shift than a single crop.
    """
    w, h = pil_image.size
    crop_ratio = 0.5
    crop_w, crop_h = int(w * crop_ratio), int(h * crop_ratio)

    crops = []
    # Center crop
    left, top = (w - crop_w)//2, (h - crop_h)//2
    crops.append(pil_image.crop((left, top, left+crop_w, top+crop_h)))
    # Four corner-ish crops (still fairly zoomed in, not edges/background)
    offsets = [
        (int(w*0.1), int(h*0.1)),
        (int(w*0.4), int(h*0.1)),
        (int(w*0.1), int(h*0.4)),
        (int(w*0.4), int(h*0.4)),
    ]
    for ox, oy in offsets[:num_crops-1]:
        ox, oy = min(ox, w-crop_w), min(oy, h-crop_h)
        crops.append(pil_image.crop((ox, oy, ox+crop_w, oy+crop_h)))

    all_preds = []
    for crop in crops:
        resized = crop.resize((224, 224))
        arr = np.expand_dims(np.array(resized), axis=0)
        preds = model.predict(arr, verbose=0)
        all_preds.append(preds[0])

    avg_preds = np.mean(all_preds, axis=0)
    predicted_class = class_names[np.argmax(avg_preds)]
    confidence = np.max(avg_preds) * 100

    return predicted_class, confidence, avg_preds

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom((-0.4, 0.3)),   # negative = zoom OUT (wider view), positive = zoom IN
    layers.RandomTranslation(0.15, 0.15),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2),
], name="data_augmentation")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import io
from PIL import Image

upload_widget = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False, description='Upload Image')
output = widgets.Output()

def on_upload_change(change):
    with output:
        clear_output()
        if len(upload_widget.value) == 0:
            return

        uploaded_file = list(upload_widget.value.values())[0] if isinstance(upload_widget.value, dict) else upload_widget.value[0]
        content = uploaded_file['content'] if isinstance(uploaded_file, dict) else uploaded_file.content

        img = Image.open(io.BytesIO(content)).convert('RGB')

        plt.figure(figsize=(5,5))
        plt.imshow(img)
        plt.axis('off')
        plt.title("Uploaded Image")
        plt.show()

        predicted_class, confidence, avg_preds = predict_with_tta(img, best_model, class_names)

        print(f"\nPredicted Fabric: {predicted_class}")
        print(f"Confidence: {confidence:.2f}%")

        top3_idx = np.argsort(avg_preds)[-3:][::-1]
        print("\nTop-3 Predictions:")
        for idx in top3_idx:
            print(f"  {class_names[idx]}: {avg_preds[idx]*100:.2f}%")

upload_widget.observe(on_upload_change, names='value')
print("Click 'Upload Image' and select a fabric photo:")
display(upload_widget, output)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import io
from PIL import Image

upload_widget = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False, description='Upload Image')
output = widgets.Output()

def on_upload_change(change):
    with output:
        clear_output()
        if len(upload_widget.value) == 0:
            return

        uploaded_file = list(upload_widget.value.values())[0] if isinstance(upload_widget.value, dict) else upload_widget.value[0]
        content = uploaded_file['content'] if isinstance(uploaded_file, dict) else uploaded_file.content

        img = Image.open(io.BytesIO(content)).convert('RGB')

        plt.figure(figsize=(5,5))
        plt.imshow(img)
        plt.axis('off')
        plt.title("Uploaded Image")
        plt.show()

        predicted_class, confidence, avg_preds = predict_with_tta(img, best_model, class_names)

        print(f"\nPredicted Fabric: {predicted_class}")
        print(f"Confidence: {confidence:.2f}%")

        top3_idx = np.argsort(avg_preds)[-3:][::-1]
        print("\nTop-3 Predictions:")
        for idx in top3_idx:
            print(f"  {class_names[idx]}: {avg_preds[idx]*100:.2f}%")

upload_widget.observe(on_upload_change, names='value')
print("Click 'Upload Image' and select a fabric photo:")
display(upload_widget, output)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import io
from PIL import Image

upload_widget = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False, description='Upload Image')
output = widgets.Output()

def on_upload_change(change):
    with output:
        clear_output()
        if len(upload_widget.value) == 0:
            return

        uploaded_file = list(upload_widget.value.values())[0] if isinstance(upload_widget.value, dict) else upload_widget.value[0]
        content = uploaded_file['content'] if isinstance(uploaded_file, dict) else uploaded_file.content

        img = Image.open(io.BytesIO(content)).convert('RGB')

        plt.figure(figsize=(5,5))
        plt.imshow(img)
        plt.axis('off')
        plt.title("Uploaded Image")
        plt.show()

        predicted_class, confidence, avg_preds = predict_with_tta(img, best_model, class_names)

        print(f"\nPredicted Fabric: {predicted_class}")
        print(f"Confidence: {confidence:.2f}%")

        top3_idx = np.argsort(avg_preds)[-3:][::-1]
        print("\nTop-3 Predictions:")
        for idx in top3_idx:
            print(f"  {class_names[idx]}: {avg_preds[idx]*100:.2f}%")

upload_widget.observe(on_upload_change, names='value')
print("Click 'Upload Image' and select a fabric photo:")
display(upload_widget, output)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import io
from PIL import Image

upload_widget = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False, description='Upload Image')
output = widgets.Output()

def on_upload_change(change):
    with output:
        clear_output()
        if len(upload_widget.value) == 0:
            return

        uploaded_file = list(upload_widget.value.values())[0] if isinstance(upload_widget.value, dict) else upload_widget.value[0]
        content = uploaded_file['content'] if isinstance(uploaded_file, dict) else uploaded_file.content

        img = Image.open(io.BytesIO(content)).convert('RGB')

        plt.figure(figsize=(5,5))
        plt.imshow(img)
        plt.axis('off')
        plt.title("Uploaded Image")
        plt.show()

        predicted_class, confidence, avg_preds = predict_with_tta(img, best_model, class_names)

        print(f"\nPredicted Fabric: {predicted_class}")
        print(f"Confidence: {confidence:.2f}%")

        top3_idx = np.argsort(avg_preds)[-3:][::-1]
        print("\nTop-3 Predictions:")
        for idx in top3_idx:
            print(f"  {class_names[idx]}: {avg_preds[idx]*100:.2f}%")

upload_widget.observe(on_upload_change, names='value')
print("Click 'Upload Image' and select a fabric photo:")
display(upload_widget, output)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import io
from PIL import Image

upload_widget = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False, description='Upload Image')
output = widgets.Output()

def on_upload_change(change):
    with output:
        clear_output()
        if len(upload_widget.value) == 0:
            return

        uploaded_file = list(upload_widget.value.values())[0] if isinstance(upload_widget.value, dict) else upload_widget.value[0]
        content = uploaded_file['content'] if isinstance(uploaded_file, dict) else uploaded_file.content

        img = Image.open(io.BytesIO(content)).convert('RGB')

        plt.figure(figsize=(5,5))
        plt.imshow(img)
        plt.axis('off')
        plt.title("Uploaded Image")
        plt.show()

        predicted_class, confidence, avg_preds = predict_with_tta(img, best_model, class_names)

        print(f"\nPredicted Fabric: {predicted_class}")
        print(f"Confidence: {confidence:.2f}%")

        top3_idx = np.argsort(avg_preds)[-3:][::-1]
        print("\nTop-3 Predictions:")
        for idx in top3_idx:
            print(f"  {class_names[idx]}: {avg_preds[idx]*100:.2f}%")

upload_widget.observe(on_upload_change, names='value')
print("Click 'Upload Image' and select a fabric photo:")
display(upload_widget, output)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import io
from PIL import Image

upload_widget = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False, description='Upload Image')
output = widgets.Output()

def on_upload_change(change):
    with output:
        clear_output()
        if len(upload_widget.value) == 0:
            return

        uploaded_file = list(upload_widget.value.values())[0] if isinstance(upload_widget.value, dict) else upload_widget.value[0]
        content = uploaded_file['content'] if isinstance(uploaded_file, dict) else uploaded_file.content

        img = Image.open(io.BytesIO(content)).convert('RGB')

        plt.figure(figsize=(5,5))
        plt.imshow(img)
        plt.axis('off')
        plt.title("Uploaded Image")
        plt.show()

        predicted_class, confidence, avg_preds = predict_with_tta(img, best_model, class_names)

        print(f"\nPredicted Fabric: {predicted_class}")
        print(f"Confidence: {confidence:.2f}%")

        top3_idx = np.argsort(avg_preds)[-3:][::-1]
        print("\nTop-3 Predictions:")
        for idx in top3_idx:
            print(f"  {class_names[idx]}: {avg_preds[idx]*100:.2f}%")

upload_widget.observe(on_upload_change, names='value')
print("Click 'Upload Image' and select a fabric photo:")
display(upload_widget, output)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import io
from PIL import Image

upload_widget = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False, description='Upload Image')
output = widgets.Output()

def on_upload_change(change):
    with output:
        clear_output()
        if len(upload_widget.value) == 0:
            return

        uploaded_file = list(upload_widget.value.values())[0] if isinstance(upload_widget.value, dict) else upload_widget.value[0]
        content = uploaded_file['content'] if isinstance(uploaded_file, dict) else uploaded_file.content

        img = Image.open(io.BytesIO(content)).convert('RGB')

        plt.figure(figsize=(5,5))
        plt.imshow(img)
        plt.axis('off')
        plt.title("Uploaded Image")
        plt.show()

        predicted_class, confidence, avg_preds = predict_with_tta(img, best_model, class_names)

        print(f"\nPredicted Fabric: {predicted_class}")
        print(f"Confidence: {confidence:.2f}%")

        top3_idx = np.argsort(avg_preds)[-3:][::-1]
        print("\nTop-3 Predictions:")
        for idx in top3_idx:
            print(f"  {class_names[idx]}: {avg_preds[idx]*100:.2f}%")

upload_widget.observe(on_upload_change, names='value')
print("Click 'Upload Image' and select a fabric photo:")
display(upload_widget, output)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import io
from PIL import Image

upload_widget = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False, description='Upload Image')
output = widgets.Output()

def on_upload_change(change):
    with output:
        clear_output()
        if len(upload_widget.value) == 0:
            return

        uploaded_file = list(upload_widget.value.values())[0] if isinstance(upload_widget.value, dict) else upload_widget.value[0]
        content = uploaded_file['content'] if isinstance(uploaded_file, dict) else uploaded_file.content

        img = Image.open(io.BytesIO(content)).convert('RGB')

        plt.figure(figsize=(5,5))
        plt.imshow(img)
        plt.axis('off')
        plt.title("Uploaded Image")
        plt.show()

        predicted_class, confidence, avg_preds = predict_with_tta(img, best_model, class_names)

        print(f"\nPredicted Fabric: {predicted_class}")
        print(f"Confidence: {confidence:.2f}%")

        top3_idx = np.argsort(avg_preds)[-3:][::-1]
        print("\nTop-3 Predictions:")
        for idx in top3_idx:
            print(f"  {class_names[idx]}: {avg_preds[idx]*100:.2f}%")

upload_widget.observe(on_upload_change, names='value')
print("Click 'Upload Image' and select a fabric photo:")
display(upload_widget, output)

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Re-run test set evaluation to check current per-class performance
y_pred_probs = best_model.predict(test_ds)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.concatenate([y for x, y in test_ds], axis=0)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Current Model')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names))

# Check exact counts for Chiffon, Wool, Velvet
for cls in ['Chiffon', 'Wool', 'Velvet']:
    idx = class_names.index(cls)
    total = cm[idx].sum()
    correct = cm[idx][idx]
    print(f"{cls}: {correct}/{total} correct ({correct/total*100:.1f}%)")
    # Show what it's being confused with
    confused_with = [(class_names[j], cm[idx][j]) for j in range(len(class_names)) if j != idx and cm[idx][j] > 0]
    confused_with.sort(key=lambda x: -x[1])
    print(f"  Confused with: {confused_with}")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for row, cls in enumerate(['Chiffon', 'Wool', 'Velvet']):
    folder = f'{train_dir}/{cls}'
    imgs = os.listdir(folder)[:4]
    for col, img_name in enumerate(imgs):
        img = Image.open(f'{folder}/{img_name}')
        axes[row][col].imshow(img)
        axes[row][col].set_title(f"{cls} #{col+1}")
        axes[row][col].axis('off')
plt.tight_layout()
plt.show()

# Check image counts
for cls in ['Chiffon', 'Wool', 'Velvet']:
    count = len(os.listdir(f'{train_dir}/{cls}'))
    print(f"{cls}: {count} training images")

In [ ]:
# Add more aggressive color/contrast augmentation to help distinguish
# texture-similar fabrics (wool/velvet) by emphasizing sheen/color differences
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom((-0.35, 0.25)),
    layers.RandomTranslation(0.15, 0.15),
    layers.RandomContrast(0.25),      # increased - helps distinguish sheen (velvet vs wool)
    layers.RandomBrightness(0.2),
    layers.RandomSaturation(0.2) if hasattr(layers, 'RandomSaturation') else layers.RandomContrast(0.1),
], name="data_augmentation")